# 9차시 프로젝트 ③ · 이력서 데이터로 채용 예측하기
> 숫자가 아니라 **채용(Hire) / 불채용(Reject)** 을 맞히는 **분류** 프로젝트 — 그리고 **AI 채용은 공정할까?**

**프로젝트 흐름** — ① 문제 정의 → ② 데이터 수집 → ③ 탐색·전처리 → ④ 모델 생성·학습 → ⑤ 모델 평가 → ⑥ 성능 개선 → ⑦ 모델 활용 → ⑧ 보고서

**진행 방법 (PRIMM)** — 🔮 **예측** → ▶️ **실행** → 👀 **관찰** → 🔧 **수정** → ✏️ **직접 만들기** / ✍️ **해석 쓰기**

**사용 데이터** (`data/` 폴더)
| 파일 | 내용 | 출처 |
|---|---|---|
| `resume_screening.csv` | 지원자 1,000명의 보유 기술·경력·학력·자격증·지원 직무·희망 연봉·프로젝트 수·AI 평가 점수·채용 결과 | Kaggle 공개 데이터 *AI-Powered Resume Screening Dataset 2025* (교재 2부 프로젝트 3, 원본 `AI_Resume_Screening.csv`) |

> ⚠️ 연습용 공개 데이터다. 실제 회사의 채용 기록이 아니며, 규칙에 따라 만들어진 것으로 보이는 패턴이 있다(③에서 확인).

## 0. 준비

도구를 불러온다. 워드클라우드(`wordcloud`)와 네트워크 그래프(`networkx`)가 없으면 설치한다.

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
try:
    import koreanize_matplotlib
except ModuleNotFoundError:
    %pip install -q koreanize-matplotlib
    import koreanize_matplotlib
try:
    from wordcloud import WordCloud
    import networkx as nx
except ModuleNotFoundError:
    %pip install -q wordcloud networkx
    from wordcloud import WordCloud
    import networkx as nx

plt.rcParams['figure.figsize'] = (6, 3.5)
DATA = 'data' if os.path.exists('data') else '../data'

---
# ① 문제 정의
**배경** 인기 있는 회사에는 지원서가 수천 장씩 들어온다. 일부 회사는 AI로 서류를 먼저 걸러 낸다.

**분석 질문** 이력서 정보(경력·프로젝트 수 등)로 채용 여부를 예측할 수 있을까? 어떤 정보가 결과와 관계가 깊을까?

**머신러닝 문제로 바꾸기**
- 입력 **X** = 경력(년), 프로젝트 수 (교재) → ⑥에서 다른 특성도 시험
- 정답 **y** = 채용 결과 (Hire = 1, Reject = 0)
- 정답이 **종류**(둘 중 하나)이므로 → **지도학습 · 분류** (6차시 다이아몬드 고가/일반과 같은 종류)

✍️ **해석 쓰기** — AI가 이력서를 먼저 걸러 낸다면 좋은 점과 걱정되는 점은 각각 무엇일까? 지원자의 입장에서도 생각해 보자.

> (여기에 한두 문장으로 적어 보세요)

---
# ② 데이터 수집
교재는 `kagglehub`로 Kaggle에서 바로 내려받는다. 여기서는 같은 파일을 `data/`에 복사해 두었다.

✏️ 빈칸 채우기: 파일을 읽고 크기를 확인한다.

In [ ]:
df = pd.read_csv(f'{DATA}/resume_screening.csv')
print('(행, 열) =', df.___)
df.head(3)

▶️ 열 이름을 우리말로 바꾼다.

In [ ]:
df.columns = ['ID', '이름', '보유기술', '경력', '최종학력', '자격증', '지원직무',
              '채용결과', '희망연봉', '프로젝트수', 'AI평가점수']
df.head(3)

---
# ③ 데이터 탐색 및 전처리
## 3-1. 기본 정리

▶️ 예측에 쓰지 않을 `ID`와 `이름`을 지운다.

In [ ]:
df = df.drop(['ID', '이름'], axis=1)
df.info()

✍️ **해석 쓰기** — 이름을 지우는 이유는 무엇일까? 이름이 모델에 들어가면 어떤 문제가 생길 수 있을까? (힌트: 이름으로 짐작할 수 있는 것들)

> (여기에 한두 문장으로 적어 보세요)

✏️ 빈칸 채우기: 결측치를 센다. 자격증의 결측치는 어떤 의미일까?

In [ ]:
df.isna().___

▶️ 자격증이 비어 있는 것은 '자격증이 없다'는 정보이므로 `'None'`이라는 범주로 채운다.

In [ ]:
df['자격증'] = df['자격증'].fillna('None')
df['자격증'].value_counts()

🔮 예측: 채용과 불채용 비율은 비슷할까? ▶️ 실행해서 확인한다.

In [ ]:
print(df['채용결과'].value_counts())
print('채용 비율:', round((df['채용결과'] == 'Hire').mean(), 3))

👀 **관찰** — 채용 812명, 불채용 188명으로 **불균형**하다. 모두 '채용'이라고 찍기만 해도 정확도가 81.2%가 나온다. ⑤에서 이 숫자를 기준선으로 쓴다.

## 3-2. 보유 기술 탐색 — 글자 데이터 다루기

✏️ 빈칸 채우기: 쉼표로 이어진 기술 목록을 쪼개(`split`) 한 줄에 하나씩 펼친(`explode`) 뒤 개수를 센다.

In [ ]:
skills = df['보유기술'].str.split(', ').explode()
skills.___.head(10)

▶️ 채용된 지원자의 기술 개수로 워드클라우드를 그린다. 많이 나온 기술일수록 글자가 크다. (`C++`처럼 기호가 있는 기술도 그대로 보이도록 개수를 직접 넣는다)

In [ ]:
hired_counts = df.loc[df['채용결과'] == 'Hire', '보유기술'].str.split(', ').explode().value_counts()
wc = WordCloud(width=600, height=300, background_color='white').generate_from_frequencies(hired_counts)
plt.imshow(wc)
plt.axis('off')
plt.title('채용된 지원자의 보유 기술')
plt.show()

▶️ 기술 네트워크: 한 사람이 함께 가진 기술끼리 선으로 잇는다(10번 이상 함께 나온 쌍만).

In [ ]:
from itertools import combinations
from collections import Counter
pairs = Counter()
for s in df['보유기술'].str.split(', '):
    for a, b in combinations(sorted(s), 2):
        pairs[(a, b)] += 1
G = nx.Graph()
for (a, b), w in pairs.items():
    if w >= 10:
        G.add_edge(a, b, weight=w)
plt.figure(figsize=(7, 5))
nx.draw(G, nx.spring_layout(G, seed=3, k=1.5), with_labels=True, node_size=500, node_color='#F4A259', font_size=8)
plt.title('함께 가진 기술 네트워크')
plt.show()

✍️ **해석 쓰기** — 네트워크 그래프에서 기술들이 몇 개의 무리로 나뉘는가? 각 무리는 어떤 직무와 관련 있어 보이는가?

> (여기에 한두 문장으로 적어 보세요)

## 3-3. 무엇이 채용과 관계가 있을까? — 교차표 (`crosstab`)

✏️ 빈칸 채우기: 경력별 채용 비율을 교차표로 만든다. `normalize='index'`는 행마다 비율로 바꾼다. 소수 둘째 자리까지 보이려면?

In [ ]:
ct = pd.crosstab(df['경력'], df['채용결과'], normalize='index')
ct.round(___)

▶️ 경력별 채용 비율을 막대그래프로 그린다.

In [ ]:
ct['Hire'].plot(kind='bar', color='#2A9D8F')
plt.title('경력별 채용 비율')
plt.xlabel('경력(년)'); plt.ylabel('채용 비율')
plt.show()

✏️ 빈칸 채우기: 같은 방법으로 프로젝트 수별 채용 비율을 본다.

In [ ]:
pd.crosstab(df[___], df['채용결과'], normalize='index')['Hire'].plot(marker='o', color='#E76F51')
plt.title('프로젝트 수별 채용 비율')
plt.ylim(0, 1.05)
plt.xlabel('프로젝트 수'); plt.ylabel('채용 비율')
plt.show()

▶️ 범주형 특성(최종학력·자격증·지원직무)도 비교한다.

In [ ]:
for col in ['최종학력', '자격증', '지원직무']:
    print(pd.crosstab(df[col], df['채용결과'], normalize='index')['Hire'].round(3), '\n')

▶️ 경력 × 프로젝트 수 조합별 채용 비율을 히트맵으로 그린다.

In [ ]:
rate = df.assign(채용=(df['채용결과'] == 'Hire').astype(int)).pivot_table(
    index='경력', columns='프로젝트수', values='채용', aggfunc='mean')
sns.heatmap(rate, cmap='RdYlGn', vmin=0, vmax=1)
plt.title('경력 × 프로젝트 수별 채용 비율')
plt.show()

✍️ **해석 쓰기** — 경력 5년 이상은 몇 %가 채용되었나? 학력·자격증·직무에 따른 차이는 큰가 작은가? 경력이 짧아도 채용된 지원자는 어떤 특징이 있나?

> (여기에 한두 문장으로 적어 보세요)

▶️ AI평가점수와 채용 결과의 관계를 본다.

In [ ]:
print(df.groupby('채용결과')['AI평가점수'].agg(['min', 'max']))

👀 **관찰** — AI평가점수 65점 이상은 **전원 채용**, 60점 이하는 **전원 불채용**이다. 이 점수는 이미 '다른 AI가 내린 판정'에 가깝다. ⑥-3에서 이 점수를 특성으로 넣으면 어떻게 되는지, 그리고 그래도 되는지 생각해 본다.

---
# ④ 모델 생성 및 학습
교재와 같이 **경력·프로젝트 수**로 **로지스틱 회귀**(이름은 회귀지만 분류 모델, 6차시) 모델을 만든다.

✏️ 빈칸 채우기: 정답을 Hire = 1, Reject = 0인 숫자로 만든다.

In [ ]:
X = df[['경력', '프로젝트수']]
y = (df['채용결과'] == 'Hire').astype(int)
y.___

▶️ 학습용 80% / 평가용 20%로 나눈다. `stratify=y`는 두 묶음의 채용 비율을 똑같이 맞춘다. 그다음 표준화한다. (6차시)

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, stratify=y, random_state=0)
scaler = StandardScaler()
X_train_s = scaler.fit_transform(X_train)
X_test_s = scaler.transform(X_test)
print('학습용:', len(X_train), '/ 평가용:', len(X_test), '/ 평가용 채용 비율:', round(y_test.mean(), 3))

✏️ 빈칸 채우기: `class_weight='balanced'`는 적은 쪽(불채용)을 틀렸을 때 더 크게 벌을 주어, 불균형을 보정한다.

In [ ]:
from sklearn.linear_model import LogisticRegression
model = LogisticRegression(class_weight='balanced', max_iter=1000, random_state=0)
model.fit(X_train_s, y_train)
print('기울기(표준화된 단위):', model.___.round(2))

✍️ **해석 쓰기** — 두 특성의 기울기가 모두 양수다. 무슨 뜻일까? 어느 특성의 영향이 더 커 보이나? (표준화했으므로 크기를 비교할 수 있다)

> (여기에 한두 문장으로 적어 보세요)

---
# ⑤ 모델 평가
분류에서는 **정확도**만 보면 안 된다. 불균형 데이터에서는 '모두 채용'이라고 찍어도 81.2%가 나오기 때문이다.

✏️ 빈칸 채우기: 평가용을 예측하고 정확도를 기준선과 비교한다.

In [ ]:
from sklearn.metrics import accuracy_score
y_pred = model.predict(X_test_s)
print('모델 정확도      :', round(___(y_test, y_pred), 3))
print('모두 채용 찍기 정확도:', round(y_test.mean(), 3))

▶️ 혼동 행렬로 무엇을 무엇으로 헷갈렸는지 본다. (행 = 실제, 열 = 예측)

In [ ]:
from sklearn.metrics import confusion_matrix
cm = confusion_matrix(y_test, y_pred, labels=[0, 1])
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=['예측: 불채용', '예측: 채용'], yticklabels=['실제: 불채용', '실제: 채용'])
plt.title('혼동 행렬')
plt.show()

**정밀도와 재현율** — 채용(1)을 기준으로
- **재현율(recall)**: 실제 채용될 사람 중 모델이 '채용'이라고 맞힌 비율 → 놓친 인재가 적을수록 높다
- **정밀도(precision)**: 모델이 '채용'이라고 한 사람 중 실제 채용된 비율 → 잘못 뽑은 사람이 적을수록 높다
- **F1**: 둘의 조화평균(균형 점수)

✏️ 빈칸 채우기: 분류 보고서로 클래스별 정밀도·재현율·F1을 한 번에 본다.

In [ ]:
from sklearn.metrics import classification_report
print(___(y_test, y_pred, target_names=['불채용(0)', '채용(1)']))

▶️ ROC-AUC: 모델이 '채용될 사람에게 더 높은 확률을 주는지'를 0.5(찍기)~1(완벽)로 나타낸다.

In [ ]:
from sklearn.metrics import roc_auc_score
y_prob = model.predict_proba(X_test_s)[:, 1]
print('ROC-AUC:', round(roc_auc_score(y_test, y_prob), 4))

✍️ **해석 쓰기** — 이 모델은 불채용 38명은 모두 맞혔지만, 실제 채용될 13명을 '불채용'으로 잘못 걸렀다. 채용에서 이 두 가지 실수 중 어느 쪽이 더 문제일까? 회사와 지원자 입장에서 각각 생각해 보자.

> (여기에 한두 문장으로 적어 보세요)

---
# ⑥ 모델 성능 개선하기
## 6-1. 🔧 `class_weight`를 빼면?

🔮 예측: 불균형 보정을 빼면 혼동 행렬이 어떻게 바뀔까? ▶️ 실행한다.

In [ ]:
def evaluate(m, Xtr, Xte, name):
    m.fit(Xtr, y_train)
    p = m.predict(Xte)
    tn, fp, fn, tp = confusion_matrix(y_test, p, labels=[0, 1]).ravel()
    return {'모델': name, '정확도': round(accuracy_score(y_test, p), 3),
            '불채용 맞힘(/38)': tn, '놓친 인재': fn, '잘못 뽑음': fp}

rows = [evaluate(LogisticRegression(class_weight='balanced', max_iter=1000, random_state=0), X_train_s, X_test_s, '보정 O (교재)'),
        evaluate(LogisticRegression(max_iter=1000, random_state=0), X_train_s, X_test_s, '보정 X')]
pd.DataFrame(rows)

## 6-2. 🔧 기준선(임계값) 옮기기
`predict`는 채용 확률이 **0.5 이상**이면 채용으로 판정한다. 이 기준을 옮기면 '놓친 인재'와 '잘못 뽑음'의 균형이 바뀐다.

✏️ 빈칸 채우기: 임계값을 0.3 ~ 0.7로 바꿔 가며 결과를 비교한다.

In [ ]:
th_rows = []
for th in [0.3, 0.5, 0.7]:
    p = (y_prob >= ___).astype(int)
    tn, fp, fn, tp = confusion_matrix(y_test, p, labels=[0, 1]).ravel()
    th_rows.append({'임계값': th, '정확도': round(accuracy_score(y_test, p), 3), '놓친 인재': fn, '잘못 뽑음': fp})
pd.DataFrame(th_rows)

✍️ **해석 쓰기** — 임계값을 낮추면 놓친 인재는 줄고 잘못 뽑는 사람은 늘 수 있다. 서류 1차 검토(뒤에 면접이 있음)라면 임계값을 어느 쪽으로 정하는 게 좋을까?

> (여기에 한두 문장으로 적어 보세요)

## 6-3. 🔧 특성 바꾸기 — 더 많이 넣으면 더 좋아질까?

▶️ 경력만 / 교재(경력·프로젝트 수) / 학력·자격증·직무 추가를 비교한다. 범주형은 원-핫 인코딩(6차시)한다.

In [ ]:
from sklearn.pipeline import make_pipeline
from sklearn.model_selection import StratifiedKFold, cross_val_score
onehot = pd.get_dummies(df[['최종학력', '자격증', '지원직무']], drop_first=True).astype(int)
feature_sets = {'경력만': df[['경력']],
                '교재: 경력·프로젝트수': df[['경력', '프로젝트수']],
                '+ 학력·자격증·직무': pd.concat([df[['경력', '프로젝트수']], onehot], axis=1)}
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=0)
for name, Xf in feature_sets.items():
    pipe = make_pipeline(StandardScaler(), LogisticRegression(class_weight='balanced', max_iter=1000, random_state=0))
    f1 = cross_val_score(pipe, Xf, y, cv=skf, scoring='f1_macro').mean()
    print(f'{name:18s} 특성 {Xf.shape[1]:2d}개 → 교차 검증 F1(평균) {f1:.3f}')

🔮 예측 → ▶️ 실행: AI평가점수를 특성에 넣으면?

In [ ]:
X_ai = df[['경력', '프로젝트수', 'AI평가점수']]
pipe = make_pipeline(StandardScaler(), LogisticRegression(class_weight='balanced', max_iter=1000, random_state=0))
print('AI평가점수 포함 → 교차 검증 F1:', round(cross_val_score(pipe, X_ai, y, cv=skf, scoring='f1_macro').mean(), 3))
print('AI평가점수 포함 → ROC-AUC     :', round(cross_val_score(pipe, X_ai, y, cv=skf, scoring='roc_auc').mean(), 4))

✍️ **해석 쓰기** — AI평가점수를 넣으면 거의 완벽해진다. 그런데 이 점수는 '다른 AI가 이미 내린 판정'에 가깝다. 이 점수를 써서 만든 모델은 무엇을 배운 걸까? 너무 좋은 점수는 왜 의심해야 할까?

> (여기에 한두 문장으로 적어 보세요)

## 6-4. 🔧 설명할 수 있는 모델: 결정 트리
결정 트리는 '경력이 몇 년 이하면 → ...'처럼 **사람이 읽을 수 있는 규칙**을 만든다. 채용처럼 사람에게 영향을 주는 결정은 **이유를 설명할 수 있어야** 한다.

✏️ 빈칸 채우기: 깊이 2짜리 결정 트리를 학습하고 규칙을 글로 출력한다.

In [ ]:
from sklearn.tree import DecisionTreeClassifier, export_text
tree = DecisionTreeClassifier(max_depth=2, random_state=0)
tree.fit(X_train, y_train)
print(export_text(tree, feature_names=['경력', '프로젝트수']))
print('정확도:', round(accuracy_score(y_test, tree.___(X_test)), 3))

✍️ **해석 쓰기** — 트리가 찾은 첫 번째 규칙은 무엇인가? 이 규칙이 3-3의 히트맵과 어떻게 연결되는가? 이렇게 단순한 규칙으로 잘 나뉜다는 것은 이 데이터에 대해 무엇을 말해 줄까?

> (여기에 한두 문장으로 적어 보세요)

---
# ⑦ 모델 활용 — 새 지원자 예측
판정(0/1)만이 아니라 **채용 확률**을 함께 본다. 확률이 0.5 근처라면 모델도 확신이 없다는 뜻이다.

✏️ 빈칸 채우기: 새 지원자 세 명의 채용 확률을 예측한다.

In [ ]:
applicants = pd.DataFrame({'경력': [3, 4, 2], '프로젝트수': [1, 8, 5]}, index=['지원자 A', '지원자 B', '지원자 C'])
applicants_s = scaler.transform(applicants)
applicants['판정'] = np.where(model.predict(applicants_s) == 1, '채용', '불채용')
applicants['채용 확률'] = model.___(applicants_s)[:, 1].round(3)
applicants

✍️ **해석 쓰기** — 지원자 C(채용 확률 약 0.42)는 '불채용'으로 판정되었다. 이 지원자를 모델 판정만으로 떨어뜨려도 될까? AI 채용 도구를 쓴다면 사람은 어떤 역할을 해야 할까?

> (여기에 한두 문장으로 적어 보세요)

---
# ✏️ 직접 만들기 — 트리 깊이 바꾸기 & 내 지원자 예측

✏️ M1: 결정 트리의 깊이를 1~5로 바꿔 정확도와 '놓친 인재' 수를 비교한다.

In [ ]:
depth_rows = []
for d in [1, 2, 3, 4, 5]:
    t = DecisionTreeClassifier(max_depth=d, random_state=0).fit(X_train, y_train)
    p = t.predict(X_test)
    tn, fp, fn, tp = confusion_matrix(y_test, p, labels=[0, 1]).ravel()
    depth_rows.append({'깊이': d, '정확도': round(accuracy_score(y_test, p), 3), '놓친 인재': fn, '잘못 뽑음': fp})
pd.DataFrame(depth_rows)

✏️ M2: 내가 정한 지원자(경력·프로젝트 수)의 채용 확률을 로지스틱 회귀로 예측한다.

In [ ]:
me = pd.DataFrame({'경력': [___], '프로젝트수': [6]})   # 숫자를 바꿔 보기
round(model.predict_proba(scaler.transform(me))[0, 1], 3)

✍️ **해석 쓰기** — M1에서 깊이가 깊을수록 항상 좋아졌나? 설명하기 쉬운 모델과 점수가 조금 더 높은 모델 중 채용에는 어느 쪽이 더 알맞을까?

> (여기에 한두 문장으로 적어 보세요)

---
# ⑧ 프로젝트 보고서 정리
| 항목 | 내용 |
|---|---|
| 문제 정의 | 어떤 질문? 누가 쓸까? 분류인 이유는? |
| 데이터 | 출처 · 크기 · 채용/불채용 비율 |
| 전처리 | 무엇을 지웠고(이름!) 결측치는 어떻게? |
| 탐색 결과 | 채용과 관계 깊은 특성 · 관계없는 특성 |
| 모델과 평가 | 정확도 · 기준선 · 혼동 행렬 · 정밀도/재현율 |
| 개선 | class_weight · 임계값 · 특성 · 결정 트리에서 알게 된 것 |
| 활용 제안 | 누구에게 어떻게? 사람은 어떤 역할? |
| 한계와 윤리 | 연습용(규칙 기반으로 보이는) 데이터 · AI평가점수 문제 · 편향 · 설명 가능성 |